## Zero Classification Shot

Dataset: Labelled Text Data

In [1]:
import os
import numpy as np
import torch
from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer
)
from sklearn.metrics import f1_score, accuracy_score

/home/provira/anaconda3/envs/spark_py3.9/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:

# ================= CONFIG =================
MODEL_NAME = "bert-base-multilingual-cased"  
DATA_PATH = "/home/provira/Documents/TFM/TFM/data/raw/Kaggle/csv/GoEmotions/goemotions_1.csv"
OUTPUT_DIR = "./emotion_model"
MAX_LENGTH = 128
BATCH_SIZE = 8
EPOCHS = 3
LR = 2e-5
SEED = 42
TEST_SIZE = 0.2
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
# ===========================================


In [3]:

# Semilla reproducible
def set_seed(seed=SEED):
    import random
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed()

print(f"Device: {DEVICE}")


Device: cuda


In [4]:

# ================= CARGAR DATASET =================
raw = load_dataset("csv", data_files=DATA_PATH)["train"]

# Columnas que NO son emociones
non_emotion_cols = {"text", "id", "author", "subreddit", "link_id", "parent_id",
                    "created_utc", "rater_id", "example_very_unclear"}

# Detectar columnas de emociones (todas menos "text")
label_columns = [c for c in raw.column_names if c not in non_emotion_cols]
num_labels = len(label_columns)
print("Etiquetas:", label_columns)

# Asegurar tipos float/int
def cast_labels(example):
    for c in label_columns:
        example[c] = float(example[c])
    return example

raw = raw.map(cast_labels)

# Split train/test
split = raw.train_test_split(test_size=TEST_SIZE, seed=SEED)
train_ds = split["train"]
eval_ds = split["test"]

Etiquetas: ['admiration', 'amusement', 'anger', 'annoyance', 'approval', 'caring', 'confusion', 'curiosity', 'desire', 'disappointment', 'disapproval', 'disgust', 'embarrassment', 'excitement', 'fear', 'gratitude', 'grief', 'joy', 'love', 'nervousness', 'optimism', 'pride', 'realization', 'relief', 'remorse', 'sadness', 'surprise', 'neutral']


In [5]:
# ================= TOKENIZACIÓN =================
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize_fn(batch):
    return tokenizer(batch["text"], truncation=True, padding="max_length", max_length=MAX_LENGTH)

train_ds = train_ds.map(tokenize_fn, batched=True)
eval_ds = eval_ds.map(tokenize_fn, batched=True)

# Seleccionar columnas a usar
cols_to_return = ["input_ids", "attention_mask"] + label_columns
train_ds.set_format(type="torch", columns=cols_to_return)
eval_ds.set_format(type="torch", columns=cols_to_return)

# ================= MODELO =================
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=num_labels,
    problem_type="multi_label_classification"
)

# ================= MÉTRICAS =================
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    probs = torch.sigmoid(torch.tensor(logits)).numpy()
    preds = (probs > 0.5).astype(int)
    labels = labels.astype(int)

    f1_micro = f1_score(labels, preds, average="micro", zero_division=0)
    f1_macro = f1_score(labels, preds, average="macro", zero_division=0)
    acc = accuracy_score(labels, preds)

    return {
        "accuracy": acc,
        "f1_micro": f1_micro,
        "f1_macro": f1_macro
    }


Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-multilingual-cased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [6]:
import transformers
print(transformers.__version__)  # should be >= 4.24

import sys
print(sys.executable)


4.55.1
/home/provira/anaconda3/envs/spark_py3.9/bin/python


In [7]:
# ================= TRAINING ARGS =================
training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    evaluation_strategy="epoch",
    save_strategy="epoch",
    logging_strategy="steps",
    logging_steps=50,
    learning_rate=LR,
    load_best_model_at_end=True,
    metric_for_best_model="f1_micro",
    save_total_limit=2,
    seed=SEED
)


# ================= TRAINER =================
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=eval_ds,
    tokenizer=tokenizer,
    compute_metrics=compute_metrics
)

# ================= ENTRENAR =================
trainer.train()

TypeError: __init__() got an unexpected keyword argument 'evaluation_strategy'

In [ ]:

# ================= GUARDAR =================
trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print(f"✅ Modelo guardado en {OUTPUT_DIR}")

# ================= PRUEBA RÁPIDA =================
from transformers import pipeline

clf = pipeline("text-classification", model=OUTPUT_DIR, tokenizer=OUTPUT_DIR, return_all_scores=True)

texto = "Odio madrugar pero amo ver el amanecer"
res = clf(texto)

print("\n📊 Resultados:")
for r in res[0]:
    print(f"{r['label']}: {r['score']:.2f}")